# 🧮 Math LLM — Build a Language Model for Mathematics from Scratch

This notebook walks you through building, training, and evaluating a
**decoder-only GPT-style Transformer** that solves mathematical problems.

**Everything is implemented from scratch** using PyTorch primitives.
No pretrained weights. No LLM APIs.

---
**Before running:** `Runtime → Change runtime type → T4 GPU`

**Estimated time:** ~30–60 minutes on T4 for the baseline (20 K steps, Stage 1).

## Step 1 — Install dependencies

In [ ]:
!pip install -q datasets==2.19.2 huggingface_hub==0.23.4
print('Dependencies installed.')

## Step 2 — Set up working directory

In [ ]:
import os, sys
REPO_PATH = '/content/math-llm'
os.makedirs(f'{REPO_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{REPO_PATH}/results', exist_ok=True)
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
print(f'Working directory: {os.getcwd()}')

## Step 3 — Verify GPU

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU. Training will be very slow.')
    DEVICE = 'cpu'
print(f'Device          : {DEVICE}')

## Step 4 — Write source modules to disk
Each `%%writefile` cell writes one module. They must all run before Step 5.

In [ ]:
%%writefile config.py
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ModelConfig:
    vocab_size: int = 256
    max_seq_len: int = 256
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 4
    d_ff: int = 512
    dropout: float = 0.1
    weight_decay: float = 0.01
    tie_weights: bool = True
    name: str = 'small-2M'

@dataclass
class TrainConfig:
    batch_size: int = 64
    learning_rate: float = 3e-4
    betas: tuple = (0.9, 0.95)
    eps: float = 1e-8
    warmup_steps: int = 200
    max_steps: int = 20_000
    min_lr_ratio: float = 0.1
    grad_clip: float = 1.0
    checkpoint_dir: str = 'checkpoints'
    log_every_steps: int = 100
    eval_every_steps: int = 500
    patience: int = 5
    seed: int = 42
    device: str = 'cuda'
    use_amp: bool = True

@dataclass
class DataConfig:
    # Primary HuggingFace source: per-category configs of the DeepMind dataset.
    # Each category is loaded as load_dataset('deepcode-ai/math_dataset', category).
    # Schema: question (str), answer (str)  — NO category column in the raw data;
    # we inject the category name ourselves.
    hf_dataset_name: str = 'deepcode-ai/math_dataset'
    # Fallback flat dataset (has question, answer, category columns).
    hf_dataset_flat: str = 'WillHeld/deepmind-math'
    categories_stage1: List[str] = field(default_factory=lambda: [
        'arithmetic__add_or_sub', 'arithmetic__mul_or_div', 'arithmetic__mixed'])
    categories_stage2: List[str] = field(default_factory=lambda: [
        'arithmetic__add_or_sub_in_base', 'arithmetic__nearest_integer_root',
        'comparison__closest', 'comparison__kth_biggest', 'comparison__sort'])
    categories_stage3: List[str] = field(default_factory=lambda: [
        'algebra__linear_1d', 'algebra__linear_2d', 'algebra__polynomial_roots',
        'measurement__conversion', 'probability__swr_p_sequence'])
    active_categories: List[str] = field(default_factory=lambda: [
        'arithmetic__add_or_sub', 'arithmetic__mul_or_div', 'arithmetic__mixed'])
    train_ratio: float = 0.80
    val_ratio: float = 0.10
    test_ratio: float = 0.10
    max_samples_per_category: int = 5_000
    max_seq_len: int = 256

def get_model_config(name='small-2M'):
    configs = {
        'tiny-1M':    ModelConfig(d_model=64,  n_heads=2, n_layers=3, d_ff=256,  name='tiny-1M'),
        'small-2M':   ModelConfig(d_model=128, n_heads=4, n_layers=4, d_ff=512,  name='small-2M'),
        'medium-5M':  ModelConfig(d_model=256, n_heads=4, n_layers=6, d_ff=1024, name='medium-5M'),
        'large-10M':  ModelConfig(d_model=384, n_heads=6, n_layers=8, d_ff=1536, max_seq_len=384, name='large-10M'),
        'xlarge-20M': ModelConfig(d_model=512, n_heads=8, n_layers=10,d_ff=2048, max_seq_len=512, dropout=0.05, name='xlarge-20M'),
    }
    return configs[name]

def get_train_config(preset='default'):
    return {
        'default':  TrainConfig(),
        'fast':     TrainConfig(batch_size=128, learning_rate=5e-4, max_steps=5_000,  warmup_steps=100),
        'thorough': TrainConfig(batch_size=64,  learning_rate=3e-4, max_steps=50_000, warmup_steps=500, patience=10),
    }[preset]
print('config.py written.')

In [ ]:
%%writefile tokenizer.py
import json, os
from typing import List, Dict, Optional

class MathTokenizer:
    SPECIAL_TOKENS = ['<PAD>', '<UNK>', '<EOS>', '<Q>', '<A>', '<BOS>']

    def __init__(self):
        self.token_to_id: Dict[str, int] = {}
        self.id_to_token: Dict[int, str] = {}
        self._built = False

    def build(self, texts: List[str]):
        self.token_to_id = {}
        self.id_to_token = {}
        for idx, tok in enumerate(self.SPECIAL_TOKENS):
            self.token_to_id[tok] = idx
            self.id_to_token[idx] = tok
        seen = set()
        for t in texts: seen.update(t)
        nid = len(self.SPECIAL_TOKENS)
        for ch in sorted(seen):
            if ch not in self.token_to_id:
                self.token_to_id[ch] = nid
                self.id_to_token[nid] = ch
                nid += 1
        self._built = True
        print(f'[Tokenizer] vocab_size={len(self.token_to_id)}')
        return self

    def _split_with_specials(self, text):
        specials = sorted(self.SPECIAL_TOKENS, key=len, reverse=True)
        result, i = [], 0
        while i < len(text):
            matched = False
            for sp in specials:
                if text[i:i+len(sp)] == sp:
                    result.append(sp); i += len(sp); matched = True; break
            if not matched:
                result.append(text[i]); i += 1
        return result

    def encode(self, text, add_eos=False, max_length=None):
        assert self._built, 'Call build() first'
        unk = self.token_to_id['<UNK>']
        ids = [self.token_to_id.get(t, unk) for t in self._split_with_specials(text)]
        if add_eos: ids.append(self.token_to_id['<EOS>'])
        if max_length: ids = ids[:max_length]
        return ids

    def decode(self, ids, skip_special_tokens=False):
        assert self._built
        sp = set(self.SPECIAL_TOKENS)
        parts = [self.id_to_token.get(i, '<UNK>') for i in ids]
        if skip_special_tokens: parts = [p for p in parts if p not in sp]
        return ''.join(parts)

    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        with open(path, 'w') as f:
            json.dump({'token_to_id': self.token_to_id,
                       'special_tokens': self.SPECIAL_TOKENS}, f, indent=2)
        print(f'[Tokenizer] Saved → {path}')

    @classmethod
    def load(cls, path):
        with open(path) as f: data = json.load(f)
        tok = cls()
        tok.token_to_id = {k: int(v) for k,v in data['token_to_id'].items()}
        tok.id_to_token  = {int(v): k for k,v in data['token_to_id'].items()}
        tok._built = True
        print(f'[Tokenizer] Loaded ← {path}  vocab_size={len(tok.token_to_id)}')
        return tok

    @property
    def vocab_size(self): return len(self.token_to_id)
    @property
    def pad_id(self): return self.token_to_id['<PAD>']
    @property
    def eos_id(self): return self.token_to_id['<EOS>']
    @property
    def a_id(self): return self.token_to_id['<A>']
    def __len__(self): return self.vocab_size
print('tokenizer.py written.')

In [ ]:
%%writefile attention.py
import math, torch, torch.nn as nn, torch.nn.functional as F
from typing import Optional

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len=256, dropout=0.1, use_flash=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.dropout_p = dropout
        self.use_flash = use_flash and hasattr(F, 'scaled_dot_product_attention')
        self.qkv_proj  = nn.Linear(d_model, 3*d_model, bias=False)
        self.out_proj  = nn.Linear(d_model, d_model,   bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop= nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer('causal_mask', mask.unsqueeze(0).unsqueeze(0))

    def forward(self, x, key_padding_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).split(self.d_model, dim=2)
        def sh(t): return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        q, k, v = sh(q), sh(k), sh(v)
        if self.use_flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                dropout_p=self.dropout_p if self.training else 0.0, is_causal=True)
        else:
            sc = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.d_k)
            sc = sc.masked_fill(~self.causal_mask[:,:,:T,:T], float('-inf'))
            if key_padding_mask is not None:
                sc = sc.masked_fill(key_padding_mask[:,None,None,:], float('-inf'))
            y = torch.matmul(self.attn_drop(torch.nan_to_num(F.softmax(sc,-1),nan=0.)), v)
        return self.resid_drop(self.out_proj(y.transpose(1,2).contiguous().view(B,T,C)))
print('attention.py written.')

In [ ]:
%%writefile transformer.py
import torch, torch.nn as nn
from attention import CausalSelfAttention

class FFN(nn.Module):
    def __init__(self, d, f, drop):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,f), nn.GELU(), nn.Dropout(drop),
                                  nn.Linear(f,d), nn.Dropout(drop))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d, h, f, L=256, drop=0.1, flash=True):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.attn = CausalSelfAttention(d, h, L, drop, flash)
        self.n2 = nn.LayerNorm(d)
        self.ffn = FFN(d, f, drop)
    def forward(self, x, mask=None):
        x = x + self.attn(self.n1(x), key_padding_mask=mask)
        return x + self.ffn(self.n2(x))

class TransformerDecoder(nn.Module):
    def __init__(self, n, d, h, f, L=256, drop=0.1, flash=True):
        super().__init__()
        self.blocks = nn.ModuleList([TransformerBlock(d,h,f,L,drop,flash) for _ in range(n)])
        self.norm   = nn.LayerNorm(d)
    def forward(self, x, mask=None):
        for b in self.blocks: x = b(x, mask)
        return self.norm(x)
print('transformer.py written.')

In [ ]:
%%writefile model.py
import torch, torch.nn as nn, torch.nn.functional as F
from config import ModelConfig
from transformer import TransformerDecoder

class MathLLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb  = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.decoder  = TransformerDecoder(cfg.n_layers, cfg.d_model, cfg.n_heads,
                                            cfg.d_ff, cfg.max_seq_len, cfg.dropout)
        self.head     = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights: self.head.weight = self.tok_emb.weight
        self._init()
        print(f'[MathLLM] {cfg.name}  params={sum(p.numel() for p in self.parameters()):,}')

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):    nn.init.normal_(m.weight, 0, 0.02)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, 0, 0.02)
            elif isinstance(m, nn.LayerNorm): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        sc = (2 * self.cfg.n_layers) ** -0.5
        for n, p in self.named_parameters():
            if 'out_proj.weight' in n: p.data.mul_(sc)

    def forward(self, ids, labels=None, mask=None):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(B, -1)
        x = self.drop(self.tok_emb(ids) + self.pos_emb(pos))
        if mask is None: mask = (ids == 0)
        logits = self.head(self.decoder(x, mask))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:,:-1].contiguous().view(-1, logits.size(-1)),
                labels[:,1:].contiguous().view(-1), ignore_index=-100)
        return {'logits': logits, 'loss': loss}

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_config_dict(self):
        return dict(self.cfg.__dict__)

    @classmethod
    def from_config_dict(cls, d):
        return cls(ModelConfig(**d))
print('model.py written.')

## Step 5 — Configuration
Change `MODEL_NAME`, `TRAIN_PRESET`, and `DATA_STAGE` here and re-run this one cell.

In [ ]:
from config import get_model_config, get_train_config, DataConfig

# ── Model size ────────────────────────────────────────────────────────────
# 'tiny-1M' → smoke-test in minutes
# 'small-2M' → default starting point
# 'medium-5M' → try this if accuracy is below target
# 'large-10M' → push for best accuracy
MODEL_NAME = 'small-2M'

# ── Training budget ───────────────────────────────────────────────────────
# 'fast'    →  5 000 steps  (~5 min T4)   — just confirms the pipeline runs
# 'default' → 20 000 steps  (~30 min T4)  — proper baseline
# 'thorough'→ 50 000 steps  (~75 min T4)  — serious accuracy attempt
TRAIN_PRESET = 'default'

# ── Data stage ────────────────────────────────────────────────────────────
# 1 = add/sub + mul/div + mixed  (simplest, trains fastest)
# 2 = stage 1 + comparison categories
# 3 = stage 2 + algebra + probability + measurement
DATA_STAGE = 1

model_cfg = get_model_config(MODEL_NAME)
train_cfg = get_train_config(TRAIN_PRESET)
data_cfg  = DataConfig()
if DATA_STAGE == 1:   data_cfg.active_categories = data_cfg.categories_stage1
elif DATA_STAGE == 2: data_cfg.active_categories = data_cfg.categories_stage1 + data_cfg.categories_stage2
else:                 data_cfg.active_categories = data_cfg.categories_stage1 + data_cfg.categories_stage2 + data_cfg.categories_stage3
train_cfg.device = DEVICE

print(f'Model    : {MODEL_NAME}')
print(f'Training : {TRAIN_PRESET}  ({train_cfg.max_steps:,} steps)')
print(f'Stage    : {DATA_STAGE}')
print(f'Categories ({len(data_cfg.active_categories)}):')
for c in data_cfg.active_categories:
    print(f'  {c}')

## Step 6 — Download dataset

We load from `deepcode-ai/math_dataset`, the canonical HuggingFace mirror of the
DeepMind Mathematics Dataset (Saxton et al. 2019).  
Schema: `question` (str), `answer` (str) — no category column; we inject it.  
If that fails we fall back to `WillHeld/deepmind-math` (flat, has `category` column).  
If both fail we use synthetic arithmetic data and **print a clear warning**.

**This cell asserts that real data was loaded and shows sample questions.**
If you see the synthetic warning, stop and check your internet connection.

In [ ]:
import hashlib, random, json
from datasets import load_dataset

# ── helpers ───────────────────────────────────────────────────────────────
def fmt(item, q='<Q>', a='<A>', eos='<EOS>'):
    return f"{q}{item['question']}{a}{item['answer']}{eos}"

def hash_bucket(text):
    return int(hashlib.md5(text.encode()).hexdigest(), 16) % 100

def do_split(by_cat, cfg):
    train, val, test = [], [], []
    tc = int(cfg.train_ratio * 100)           # 80
    vc = tc + int(cfg.val_ratio * 100)        # 90
    for cat, items in by_cat.items():
        for item in items:
            b = hash_bucket(item['question'])
            if   b < tc: train.append(item)
            elif b < vc: val.append(item)
            else:        test.append(item)
    return train, val, test

# ── source A: deepcode-ai/math_dataset (per-category config) ──────────────
def load_deepcode(cfg):
    """Load each active category as its own dataset config.
    Schema: question, answer   (no category column → inject it)"""
    by_cat = {c: [] for c in cfg.active_categories}
    rng = random.Random(42)
    for cat in cfg.active_categories:
        ds = load_dataset('deepcode-ai/math_dataset', cat, split='train',
                          trust_remote_code=True)
        items = []
        for ex in ds:
            q = str(ex['question']).strip()
            a = str(ex['answer']).strip()
            if q and a:
                items.append({'question': q, 'answer': a, 'category': cat})
        rng.shuffle(items)
        by_cat[cat] = items[:cfg.max_samples_per_category]
        print(f'  deepcode  {cat}: {len(by_cat[cat]):,} examples loaded')
    return by_cat

# ── source B: WillHeld/deepmind-math (flat, has category column) ──────────
def load_willheld(cfg):
    """Load the flat combined dataset and filter to active categories.
    Schema: question, answer, category"""
    ds = load_dataset('WillHeld/deepmind-math', split='train', trust_remote_code=True)
    by_cat = {c: [] for c in cfg.active_categories}
    rng = random.Random(42)
    for ex in ds:
        cat = ex.get('category', '')          # correct column name
        if cat not in cfg.active_categories: continue
        q = str(ex['question']).strip()
        a = str(ex['answer']).strip()
        if q and a:
            by_cat[cat].append({'question': q, 'answer': a, 'category': cat})
    for cat in cfg.active_categories:
        rng.shuffle(by_cat[cat])
        by_cat[cat] = by_cat[cat][:cfg.max_samples_per_category]
        print(f'  willheld  {cat}: {len(by_cat[cat]):,} examples loaded')
    return by_cat

# ── source C: synthetic fallback ──────────────────────────────────────────
def load_synthetic(cfg):
    rng = random.Random(42)
    n   = cfg.max_samples_per_category
    by_cat = {c: [] for c in cfg.active_categories}
    for cat in cfg.active_categories:
        items = []
        for _ in range(n):
            a, b = rng.randint(-999, 999), rng.randint(-999, 999)
            if cat == 'arithmetic__add_or_sub':
                op = '+' if rng.random() < .5 else '-'
                ans = a+b if op=='+' else a-b
                items.append({'question': f'What is {a} {op} {b}?',
                               'answer': str(ans), 'category': cat})
            elif cat == 'arithmetic__mul_or_div':
                a, b = rng.randint(1,99), rng.randint(1,99)
                if rng.random() < .5:
                    items.append({'question': f'What is {a} * {b}?',
                                   'answer': str(a*b), 'category': cat})
                else:
                    items.append({'question': f'What is {a*b} / {b}?',
                                   'answer': str(a), 'category': cat})
            elif cat == 'arithmetic__mixed':
                a, b, c = rng.randint(1,50), rng.randint(1,50), rng.randint(1,50)
                items.append({'question': f'What is {a} + {b} * {c}?',
                               'answer': str(a+b*c), 'category': cat})
            else:
                # generic add/sub for unknown categories in fallback
                items.append({'question': f'What is {a} + {b}?',
                               'answer': str(a+b), 'category': cat})
        by_cat[cat] = items
        print(f'  synthetic {cat}: {len(items):,} examples')
    return by_cat

# ── try each source in order ──────────────────────────────────────────────
DATA_SOURCE = None

try:
    print('Trying deepcode-ai/math_dataset …')
    by_category = load_deepcode(data_cfg)
    DATA_SOURCE = 'deepcode-ai/math_dataset'
except Exception as e:
    print(f'deepcode-ai failed: {e}')
    try:
        print('Trying WillHeld/deepmind-math …')
        by_category = load_willheld(data_cfg)
        DATA_SOURCE = 'WillHeld/deepmind-math'
    except Exception as e2:
        print(f'WillHeld failed: {e2}')
        print('\n' + '!'*60)
        print('WARNING: Both HuggingFace sources failed.')
        print('Using SYNTHETIC arithmetic data.')
        print('Results will reflect synthetic data quality, not the real dataset.')
        print('!'*60 + '\n')
        by_category = load_synthetic(data_cfg)
        DATA_SOURCE = 'synthetic'

print(f'\nData source: {DATA_SOURCE}')

# ── totals ────────────────────────────────────────────────────────────────
total_loaded = sum(len(v) for v in by_category.values())
print(f'Total examples loaded: {total_loaded:,}')
assert total_loaded > 0, 'FATAL: no examples loaded. Check connectivity or data config.'

# ── verify real data: spot-check first 3 examples per category ────────────
print('\n--- Sample questions from each category ---')
for cat, items in by_category.items():
    print(f'\n[{cat}]  ({len(items):,} examples)')
    for it in items[:3]:
        print(f'  Q: {it["question"]}')
        print(f'  A: {it["answer"]}')

## Step 6b — Split and verify no overlap

In [ ]:
train_items, val_items, test_items = do_split(by_category, data_cfg)

print(f'Split counts:')
print(f'  train : {len(train_items):,}')
print(f'  val   : {len(val_items):,}')
print(f'  test  : {len(test_items):,}')
print(f'  total : {len(train_items)+len(val_items)+len(test_items):,}')

# ── assert no overlap between splits ─────────────────────────────────────
train_qs = {it['question'] for it in train_items}
val_qs   = {it['question'] for it in val_items}
test_qs  = {it['question'] for it in test_items}
assert len(train_qs & test_qs)  == 0, f'LEAKAGE: {len(train_qs & test_qs)} train/test overlaps'
assert len(train_qs & val_qs)   == 0, f'LEAKAGE: {len(train_qs & val_qs)} train/val overlaps'
assert len(val_qs   & test_qs)  == 0, f'LEAKAGE: {len(val_qs & test_qs)} val/test overlaps'
print('\nSplit overlap check: PASSED (zero leakage)')

# ── assert test set is non-trivially large ────────────────────────────────
assert len(test_items) >= 100, f'Test set too small: {len(test_items)} examples'
print(f'Test set size check: PASSED ({len(test_items):,} ≥ 100)')

# ── show a few test examples (these will never be trained on) ─────────────
print('\n--- 5 sample test-set questions (held out, never trained on) ---')
for it in test_items[:5]:
    print(f'  [{it["category"]}]  Q: {it["question"]}  →  A: {it["answer"]}')

## Step 7 — Build tokenizer

In [ ]:
from tokenizer import MathTokenizer

# Build vocabulary from ALL data (train+val+test combined).
# This prevents <UNK> tokens on the test set — the tokenizer
# only learns character→id mappings, not which split an example belongs to.
all_texts = [fmt(it) for items in by_category.values() for it in items]
tokenizer = MathTokenizer()
tokenizer.build(all_texts)
model_cfg.vocab_size = tokenizer.vocab_size

print(f'Vocabulary size : {tokenizer.vocab_size}')
print(f'Special tokens  : {tokenizer.SPECIAL_TOKENS}')

# ── round-trip test ───────────────────────────────────────────────────────
# Use a question from the actual loaded data, not a hand-crafted string.
sample = fmt(train_items[0])
ids    = tokenizer.encode(sample)
back   = tokenizer.decode(ids)
assert back == sample, f'Round-trip FAILED:\n  in : {sample!r}\n  out: {back!r}'
print(f'Round-trip test : PASSED')
print(f'  example : {sample}')
print(f'  token ids (first 20): {ids[:20]}')

# ── confirm no UNK tokens appear in any split ─────────────────────────────
unk_count = sum(
    tokenizer.token_to_id['<UNK>'] in tokenizer.encode(fmt(it))
    for it in test_items
)
assert unk_count == 0, f'{unk_count} test examples contain <UNK> tokens'
print(f'UNK token check : PASSED (0 <UNK> tokens in test set)')

tokenizer.save('checkpoints/tokenizer.json')

## Step 8 — Create DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from functools import partial

class MathDataset(Dataset):
    def __init__(self, items, tokenizer, max_seq_len=256):
        self.examples = []
        a_id = tokenizer.token_to_id['<A>']
        skipped = 0
        for item in items:
            ids = tokenizer.encode(fmt(item), max_length=max_seq_len)
            if len(ids) < 4: skipped += 1; continue
            labels = ids.copy()
            a_pos  = next((i for i,x in enumerate(ids) if x==a_id), None)
            if a_pos is not None:
                for j in range(a_pos+1): labels[j] = -100
            self.examples.append({'ids': ids, 'labels': labels,
                                   'cat': item.get('category','?'),
                                   'q': item['question'], 'a': item['answer']})
        if skipped: print(f'  Skipped {skipped} too-short examples')

    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        e = self.examples[i]
        return {'input_ids': torch.tensor(e['ids'],    dtype=torch.long),
                'labels':    torch.tensor(e['labels'], dtype=torch.long),
                'category': e['cat'], 'question': e['q'], 'answer': e['a']}

def collate(batch, pad_id=0):
    ml = max(len(b['input_ids']) for b in batch)
    ids, labs = [], []
    for b in batch:
        p = ml - len(b['input_ids'])
        ids.append(torch.cat([b['input_ids'], torch.full((p,), pad_id, dtype=torch.long)]))
        labs.append(torch.cat([b['labels'],   torch.full((p,), -100,   dtype=torch.long)]))
    return {'input_ids':  torch.stack(ids),
            'labels':     torch.stack(labs),
            'categories': [b['category'] for b in batch],
            'questions':  [b['question'] for b in batch],
            'answers':    [b['answer']   for b in batch]}

_col = partial(collate, pad_id=tokenizer.pad_id)
train_ds = MathDataset(train_items, tokenizer, model_cfg.max_seq_len)
val_ds   = MathDataset(val_items,   tokenizer, model_cfg.max_seq_len)
test_ds  = MathDataset(test_items,  tokenizer, model_cfg.max_seq_len)

train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size, shuffle=True,  num_workers=0, collate_fn=_col)
val_loader   = DataLoader(val_ds,   batch_size=128,                  shuffle=False, num_workers=0, collate_fn=_col)
test_loader  = DataLoader(test_ds,  batch_size=128,                  shuffle=False, num_workers=0, collate_fn=_col)

print(f'Dataset sizes:  train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

# ── inspect one real batch ────────────────────────────────────────────────
b0 = next(iter(train_loader))
print(f'Batch shape: input_ids={tuple(b0["input_ids"].shape)}')
print(f'\nFirst training example (decoded):')
print(f'  {tokenizer.decode(b0["input_ids"][0].tolist())}')
print(f'Label mask (1=predict, 0=ignored):')
mask_display = ['1' if l != -100 else '0' for l in b0['labels'][0].tolist()]
print(f'  {".".join(mask_display[:40])}…')

# ── assert the model will actually train on real answer tokens ────────────
n_answer_tokens = (b0['labels'] != -100).sum().item()
n_total_tokens  = b0['labels'].numel()
assert n_answer_tokens > 0, 'FATAL: every label is -100 — nothing to train on'
print(f'\nAnswer tokens in first batch: {n_answer_tokens}/{n_total_tokens} '
      f'({100*n_answer_tokens/n_total_tokens:.1f}%) — PASSED')

## Step 9 — Build model and verify forward pass

In [ ]:
from model import MathLLM
import math

device = torch.device(DEVICE)
model  = MathLLM(model_cfg).to(device)
print(f'Trainable parameters: {model.num_parameters():,}')

# Forward pass on a real batch (not dummy data)
model.eval()
with torch.no_grad():
    real_ids    = b0['input_ids'][:2].to(device)
    real_labels = b0['labels'][:2].to(device)
    out = model(real_ids, labels=real_labels)
print(f'Forward pass:  logits={tuple(out["logits"].shape)}  loss={out["loss"].item():.4f}')
# Random-init loss should be close to log(vocab_size)
expected_random_loss = math.log(tokenizer.vocab_size)
print(f'Expected random-init loss ≈ log({tokenizer.vocab_size}) = {expected_random_loss:.4f}')
assert out['loss'].item() < expected_random_loss * 2, 'Loss is suspiciously high'
print('Loss sanity check: PASSED')
model.train()

## Step 10 — Train
Watch that `loss` falls and `tok_acc` rises. If loss stays flat after 500 steps,
something is wrong with the data or learning rate.

In [ ]:
import time

def cosine_lr(step, warmup, max_steps, max_lr, min_lr):
    if step < warmup: return max_lr * step / max(warmup, 1)
    if step >= max_steps: return min_lr
    p = (step - warmup) / max(max_steps - warmup, 1)
    return min_lr + 0.5*(max_lr-min_lr)*(1 + math.cos(math.pi*p))

def tok_acc(logits, labels):
    mask = labels != -100
    if not mask.any(): return 0.0
    return ((logits.argmax(-1) == labels) & mask).sum().item() / mask.sum().item()

# optimiser — no weight decay on biases / layernorm
decay, no_decay = [], []
for n, p in model.named_parameters():
    (decay if p.ndim >= 2 else no_decay).append(p)
optimizer = torch.optim.AdamW(
    [{'params': decay,    'weight_decay': model_cfg.weight_decay},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=train_cfg.learning_rate, betas=train_cfg.betas, eps=train_cfg.eps)

use_amp = (DEVICE == 'cuda') and train_cfg.use_amp
scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)
min_lr  = train_cfg.learning_rate * train_cfg.min_lr_ratio

best_val_loss = float('inf')
patience_ctr  = 0
history = {'step': [], 'val_loss': [], 'val_tok_acc': []}
step, t0 = 0, time.time()
rl, ra, rn = 0.0, 0.0, 0

def run_val():
    model.eval()
    tl, ta, nb = 0.0, 0.0, 0
    with torch.no_grad():
        for vb in val_loader:
            vi, vl = vb['input_ids'].to(device), vb['labels'].to(device)
            with torch.cuda.amp.autocast(enabled=use_amp):
                vo = model(vi, labels=vl)
            if vo['loss'] is not None:
                tl += vo['loss'].item(); ta += tok_acc(vo['logits'], vl); nb += 1
    model.train()
    return (tl/nb, ta/nb) if nb else (float('inf'), 0.0)

model.train()
print(f'Training {train_cfg.max_steps:,} steps  '
      f'batch={train_cfg.batch_size}  lr={train_cfg.learning_rate}  '
      f'data_source={DATA_SOURCE}')
print('-' * 70)

while step < train_cfg.max_steps:
    for batch in train_loader:
        if step >= train_cfg.max_steps: break
        ids    = batch['input_ids'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        lr = cosine_lr(step, train_cfg.warmup_steps, train_cfg.max_steps,
                        train_cfg.learning_rate, min_lr)
        for g in optimizer.param_groups: g['lr'] = lr
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            out  = model(ids, labels=labels)
            loss = out['loss']
        if loss is None or torch.isnan(loss): step += 1; continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        scaler.step(optimizer); scaler.update()
        rl += loss.item(); ra += tok_acc(out['logits'].detach(), labels); rn += 1
        if step % train_cfg.log_every_steps == 0 and rn > 0:
            e = time.time()-t0
            eta = e/max(step,1)*(train_cfg.max_steps-step)
            print(f'step {step:6d}  loss={rl/rn:.4f}  tok_acc={ra/rn:.3f}  '
                  f'lr={lr:.1e}  {e:.0f}s  ETA={eta:.0f}s')
            rl, ra, rn = 0.0, 0.0, 0
        if step % train_cfg.eval_every_steps == 0:
            vl, va = run_val()
            history['step'].append(step)
            history['val_loss'].append(vl)
            history['val_tok_acc'].append(va)
            is_best = vl < best_val_loss
            if is_best:
                best_val_loss = vl; patience_ctr = 0
                torch.save({'step': step, 'val_loss': vl,
                            'model_state': model.state_dict(),
                            'cfg_model': model_cfg.__dict__,
                            'tokenizer_path': 'checkpoints/tokenizer.json',
                            'data_source': DATA_SOURCE},
                           'checkpoints/best_model.pt')
                print(f'  [Val] loss={vl:.4f}  tok_acc={va:.3f}  ★ new best')
            else:
                patience_ctr += 1
                print(f'  [Val] loss={vl:.4f}  tok_acc={va:.3f}  '
                      f'patience={patience_ctr}/{train_cfg.patience}')
                if patience_ctr >= train_cfg.patience:
                    print('Early stopping.'); break
        step += 1

print(f'\nDone.  {time.time()-t0:.0f}s  best_val_loss={best_val_loss:.4f}')

# save final
torch.save({'step': step, 'val_loss': best_val_loss,
            'model_state': model.state_dict(),
            'cfg_model': model_cfg.__dict__,
            'tokenizer_path': 'checkpoints/tokenizer.json',
            'data_source': DATA_SOURCE},
           'checkpoints/final_model.pt')
with open('results/training_history.json', 'w') as f:
    json.dump({**history, 'data_source': DATA_SOURCE,
               'model': MODEL_NAME, 'stage': DATA_STAGE}, f, indent=2)
print('Checkpoints and history saved.')

## Step 11 — Training curves

In [ ]:
import matplotlib.pyplot as plt
if history['step']:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(history['step'], history['val_loss'], 'b-o', ms=4)
    ax1.set(xlabel='Step', ylabel='Loss', title='Val Loss'); ax1.grid(True)
    ax2.plot(history['step'], history['val_tok_acc'], 'g-o', ms=4)
    ax2.set(xlabel='Step', ylabel='Token Accuracy', title='Val Token Acc'); ax2.grid(True)
    plt.tight_layout()
    plt.savefig('results/training_curve.png', dpi=120)
    plt.show()
    print(f'Final val loss: {history["val_loss"][-1]:.4f}')
    print(f'Final val tok_acc: {history["val_tok_acc"][-1]:.4f}')

## Step 12 — Autoregressive generation

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def generate(model, tokenizer, question, device, max_new=64, temperature=0.0):
    model.eval()
    prompt = tokenizer.encode(f'<Q>{question}<A>')
    ids = torch.tensor([prompt], dtype=torch.long, device=device)
    gen = []
    for _ in range(max_new):
        out    = model(ids[:, -model.cfg.max_seq_len:])
        logits = out['logits'][0, -1]
        nid    = logits.argmax() if temperature == 0 else \
                 torch.multinomial(F.softmax(logits/temperature, -1), 1).squeeze()
        gen.append(nid.item())
        ids = torch.cat([ids, nid.view(1,1)], dim=1)
        if nid.item() == tokenizer.eos_id: break
    full = tokenizer.decode(prompt + gen)
    ap   = full.find('<A>')
    ep   = full.find('<EOS>', ap)
    return (full[ap+3:ep].strip() if ap != -1 else full), full

# Load best checkpoint
ckpt = torch.load('checkpoints/best_model.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best model (step={ckpt["step"]}  val_loss={ckpt["val_loss"]:.4f})')
print(f'Data source used during training: {ckpt.get("data_source", "unknown")}')

# Use questions from the test set — not hand-picked
print('\n--- Generating answers for 10 test-set questions ---')
for it in test_items[:10]:
    pred, _ = generate(model, tokenizer, it['question'], device)
    ok = '✓' if pred.strip() == it['answer'].strip() else '✗'
    print(f'  {ok}  Q: {it["question"][:60]}')
    print(f'       gold={it["answer"]!r}  pred={pred!r}')

## Step 13 — Full test-set evaluation
This is the number that matters. Every example in `test_items` was excluded from training.

In [ ]:
from collections import defaultdict

# ── token accuracy (teacher-forced, fast) ─────────────────────────────────
model.eval()
total_tok, correct_tok = 0, 0
with torch.no_grad():
    for b in test_loader:
        ids    = b['input_ids'].to(device)
        labels = b['labels'].to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(ids, labels=labels)
        mask = labels != -100
        correct_tok += ((out['logits'].argmax(-1) == labels) & mask).sum().item()
        total_tok   += mask.sum().item()
token_accuracy = correct_tok / max(total_tok, 1)
print(f'Token accuracy (teacher-forced): {token_accuracy:.4f}  ({correct_tok}/{total_tok})')

# ── exact-answer accuracy (autoregressive) ────────────────────────────────
print(f'\nEvaluating exact-answer accuracy on {len(test_items):,} test examples …')
per_cat  = defaultdict(lambda: {'n': 0, 'correct': 0})
failures = []
correct  = 0

for i, it in enumerate(test_items):
    gold = it['answer'].strip()
    pred, _ = generate(model, tokenizer, it['question'], device)
    pred = pred.strip()
    match = (pred == gold) or (' '.join(pred.lower().split()) == ' '.join(gold.lower().split()))
    correct += int(match)
    per_cat[it.get('category','?')]['n'] += 1
    per_cat[it.get('category','?')]['correct'] += int(match)
    if not match:
        failures.append({'q': it['question'], 'gold': gold, 'pred': pred,
                          'cat': it.get('category','?')})
    if (i+1) % 100 == 0 or (i+1) == len(test_items):
        print(f'  {i+1}/{len(test_items)}  running_acc={correct/(i+1):.4f}')

exact_acc = correct / len(test_items)

# ── results ────────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'EVALUATION RESULTS — held-out test set')
print(f'Data source  : {DATA_SOURCE}')
print(f'Model        : {MODEL_NAME}')
print(f'Training steps: {ckpt["step"]:,}')
print(f'{"="*60}')
print(f'Token accuracy (teacher-forced)  : {token_accuracy:.4f}')
print(f'Exact-answer accuracy (generated): {exact_acc:.4f}  ({correct}/{len(test_items)})')
print()
print('Per category:')
for cat, v in sorted(per_cat.items()):
    cat_acc = v['correct'] / max(v['n'], 1)
    print(f'  {cat:<45} {cat_acc:.4f}  ({v["correct"]}/{v["n"]})')

# ── save ──────────────────────────────────────────────────────────────────
results = {
    'data_source':            DATA_SOURCE,
    'model_name':             MODEL_NAME,
    'data_stage':             DATA_STAGE,
    'training_steps':         ckpt['step'],
    'token_accuracy':         token_accuracy,
    'exact_answer_accuracy':  exact_acc,
    'correct':                correct,
    'total_test_examples':    len(test_items),
    'per_category':           {c: {'acc': v['correct']/max(v['n'],1), **v}
                                for c,v in per_cat.items()},
    'n_failures':             len(failures),
    'sample_failures':        failures[:20],
}
with open('results/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('\nresults/eval_results.json saved.')

## Step 14 — Failure analysis and next steps

In [ ]:
print(f'Failures: {len(failures)}/{len(test_items)}  '
      f'({100*(1-exact_acc):.1f}%)')
print()
for i, f in enumerate(failures[:10]):
    print(f'  [{i+1}] {f["cat"]}')
    print(f'       Q   : {f["q"][:70]}')
    print(f'       gold: {f["gold"]!r}')
    print(f'       pred: {f["pred"]!r}')
    print()

if exact_acc < 1.0:
    print(f'Accuracy: {exact_acc:.1%}  — not yet at 100%')
    print()
    if exact_acc < 0.50:
        print('Next step: model is under-trained or data has issues.')
        print('  → Check that DATA_SOURCE is not synthetic.')
        print('  → Try medium-5M model or thorough training preset.')
    elif exact_acc < 0.80:
        print('Next step: making progress but needs more capacity or training.')
        print('  → Try medium-5M with default preset.')
        print('  → Consider DATA_STAGE=2 for more variety.')
    elif exact_acc < 0.95:
        print('Next step: good accuracy. Fine-tuning likely to help.')
        print('  → Try large-10M model with thorough preset.')
        print('  → Look at failure categories — some may need more data.')
    else:
        print('Next step: >95%. Very close to target.')
        print('  → Check failures — are they edge cases or systematic errors?')
        print('  → Extended training (thorough preset) may close the gap.')
else:
    print('100% exact-answer accuracy on the held-out test set!')
    print(f'  Verified on {len(test_items):,} genuinely unseen examples.')
    print(f'  Data source: {DATA_SOURCE}')

## Step 15 — Save to Google Drive

In [ ]:
# Uncomment to persist to Drive
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# DRIVE = '/content/drive/MyDrive/math-llm'
# shutil.copytree('checkpoints', DRIVE+'/checkpoints', dirs_exist_ok=True)
# shutil.copytree('results',     DRIVE+'/results',     dirs_exist_ok=True)
# print(f'Saved to {DRIVE}')

import os
print('checkpoints/:', os.listdir('checkpoints'))
print('results/:',     os.listdir('results'))

## Step 16 — Reload and resume without retraining

In [ ]:
import torch, json
from model import MathLLM
from config import ModelConfig
from tokenizer import MathTokenizer

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt      = torch.load('checkpoints/best_model.pt', map_location=device, weights_only=False)
m2        = MathLLM(ModelConfig(**ckpt['cfg_model'])).to(device)
m2.load_state_dict(ckpt['model_state'])
m2.eval()
tok2      = MathTokenizer.load(ckpt.get('tokenizer_path', 'checkpoints/tokenizer.json'))

print(f'Reloaded: {ckpt["cfg_model"]["name"]}  '
      f'step={ckpt["step"]}  val_loss={ckpt["val_loss"]:.4f}')
print(f'Data source: {ckpt.get("data_source", "unknown")}')

q = test_items[0]['question']
a, _ = generate(m2, tok2, q, device)
print(f'\nTest: Q={q!r}')
print(f'      pred={a!r}  gold={test_items[0]["answer"]!r}')